In [27]:
import pandas as pd

In [28]:
path_og = "./lia_gts.csv"
path_fps = "./lia_fps.csv"
path_rsna = "./rsna_gts.csv" #14 added back
path_fps_rsna = "./rsna_fps.csv"
df_fps_rsna = pd.read_csv(path_fps_rsna)

df = pd.read_csv(path_og)
df_rsna = pd.read_csv(path_rsna)
df_rsna.columns = [
    "seriesuid",
    "coordX",
    "coordY",
    "coordZ",
    "d",
    "h",
    "w",
    "lesion",
    "volume",
    "maj_axis",
    "min_axis",
]
df_final = df[df_rsna.columns]
df_final = pd.concat([df_final, df_rsna], ignore_index=True)

In [29]:
df_final.to_csv("./annots/lia_rsna.csv", index=False)

In [30]:
df_fps_og = pd.read_csv(path_fps).iloc[:, 2:20]

In [31]:
# use only the validated FPs from df_fps_og

def only_validated_fp(row):
    # if jisoo marked as aneurysm then remove
    if row["is_aneurysm"] == 1:
        return False
    # if jisoo marked as non-aneurysm then keep
    elif row["is_fp"] == "1" or row["is_fp"] == "1P" or row["is_fp"] == 1:
        return True
    # if jisoo marked as infundibulum then discard
    elif row["is_infundibulum"] == 1:
        return True
    # if either of these are NaN, we need to check the Aneurysm column
    else:
        return False 

def only_validated_aneurysm(row):
    if row["is_aneurysm"] == 1:
        return True
    else:
        return False

columns_basic = [
    "seriesuid",
    "coordX",
    "coordY",
    "coordZ",
    "d",
    "h",
    "w",
    "lesion",]
    
columns_extra = [
    "volume",
    "maj_axis",
    "min_axis",
    "iom_artery",
    "iom_vein"
]

def standardize_columns(df, columns_basic, columns_extra):
    df = df.copy()
    df = df[columns_basic]
    for col in columns_extra:
        if col not in df.columns:
            df[col] = None
    return df[columns_basic + columns_extra]

df_lia_fps_validated = df_fps_og[df_fps_og.apply(only_validated_fp, axis=1)]
df_lia_fps_validated["lesion"] = "non_aneurysm"
df_lia_fps_validated = standardize_columns(
    df_lia_fps_validated, columns_basic, columns_extra
)
df_lia_gts_validated = df_fps_og[df_fps_og.apply(only_validated_aneurysm, axis=1)]
df_lia_gts_validated["lesion"] = "aneurysm"
df_lia_gts_validated = standardize_columns(
    df_lia_gts_validated, columns_basic, columns_extra
)

df_lia_fps_gts_validated = pd.concat(
    [df_lia_fps_validated, df_lia_gts_validated], ignore_index=True
)

# merge the two dfs with df

df_lia_fps_gts_validated = pd.concat(
    [df, df_lia_fps_gts_validated], ignore_index=True
)

df_lia_fps_validated = pd.concat(
    [df, df_lia_fps_validated], ignore_index=True
)

/tmp/ipykernel_1251328/399359214.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_lia_fps_validated["lesion"] = "non_aneurysm"
/tmp/ipykernel_1251328/399359214.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_lia_gts_validated["lesion"] = "aneurysm"
/tmp/ipykernel_1251328/399359214.py:66: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. 

In [32]:
len(df_lia_fps_validated), len(df_lia_fps_gts_validated), len(df_lia_fps_validated)- len(df), len(df_lia_fps_gts_validated)- len(df)

(1766, 1857, 393, 484)

In [33]:
df_lia_fps_validated.to_csv("./annots/lia_with_fps.csv", index=False)
df_lia_fps_gts_validated.to_csv("./annots/lia_with_fps_and_gts.csv", index=False)

In [34]:
df_rsna_fps_validated = df_fps_rsna[df_fps_rsna.apply(only_validated_fp, axis=1)]
df_rsna_fps_validated["lesion"] = "non_aneurysm"
df_rsna_fps_validated = standardize_columns(
    df_rsna_fps_validated, columns_basic, columns_extra
)
df_rsna_gts_validated = df_fps_rsna[df_fps_rsna.apply(only_validated_aneurysm, axis=1)]
df_rsna_gts_validated["lesion"] = "aneurysm"
df_rsna_gts_validated = standardize_columns(
    df_rsna_gts_validated, columns_basic, columns_extra
)
df_rsna_fps_gts_validated = pd.concat(
    [df_rsna_fps_validated, df_rsna_gts_validated], ignore_index=True
)
df_rsna_fps_gts_validated = pd.concat(
    [df_rsna, df_rsna_fps_gts_validated], ignore_index=True
)
df_rsna_fps_validated = pd.concat(
    [df_rsna, df_rsna_fps_validated], ignore_index=True
)

/tmp/ipykernel_1251328/2890146260.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rsna_fps_validated["lesion"] = "non_aneurysm"
/tmp/ipykernel_1251328/2890146260.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rsna_gts_validated["lesion"] = "aneurysm"
/tmp/ipykernel_1251328/2890146260.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtype

In [35]:
len(df_rsna_fps_validated), len(df_rsna_fps_gts_validated), len(df_rsna_fps_validated)- len(df_rsna), len(df_rsna_fps_gts_validated)- len(df_rsna)

(1217, 1300, 190, 273)

In [36]:
df_rsna_std = standardize_columns(
    df_rsna, columns_basic, columns_extra
)
df_rsna_std.to_csv("annots/rsna.csv", index=False)

In [37]:
df_rsna_fps_validated.to_csv("./annots/rsna_with_fps.csv", index=False)
df_rsna_fps_gts_validated.to_csv("./annots/rsna_with_fps_and_gts.csv", index=False)

df_lia_rsna_fps_validated = pd.concat(
    [df_lia_fps_validated, df_rsna_fps_validated ], ignore_index=True
)
df_lia_rsna_fps_gts_validated = pd.concat(
    [df_lia_fps_gts_validated, df_rsna_fps_gts_validated ], ignore_index=True
)   
df_lia_rsna_fps_validated.to_csv("./annots/lia_rsna_with_fps.csv", index=False)
df_lia_rsna_fps_gts_validated.to_csv("./annots/lia_rsna_with_fps_and_gts.csv", index=False)

/tmp/ipykernel_1251328/1255673997.py:4: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_lia_rsna_fps_validated = pd.concat(
/tmp/ipykernel_1251328/1255673997.py:7: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_lia_rsna_fps_gts_validated = pd.concat(


In [38]:
# for each dataset print value counts
print("LIA only with FPs:")
print(df_lia_fps_validated['lesion'].value_counts())
print("\nLIA only with FPs and GTs:")
print(df_lia_fps_gts_validated['lesion'].value_counts())
print("\nRSNA only with FPs:")
print(df_rsna_fps_validated['lesion'].value_counts())
print("\nRSNA only with FPs and GTs:")
print(df_rsna_fps_gts_validated['lesion'].value_counts())
print("\nLIA + RSNA")
print(df_final['lesion'].value_counts())
print("\nLIA + RSNA with FPs:")
print(df_lia_rsna_fps_validated['lesion'].value_counts())
print("\nLIA + RSNA with FPs and GTs:")
print(df_lia_rsna_fps_gts_validated['lesion'].value_counts())


LIA only with FPs:
lesion
aneurysm        1373
non_aneurysm     393
Name: count, dtype: int64

LIA only with FPs and GTs:
lesion
aneurysm        1464
non_aneurysm     393
Name: count, dtype: int64

RSNA only with FPs:
lesion
aneurysm        1027
non_aneurysm     190
Name: count, dtype: int64

RSNA only with FPs and GTs:
lesion
aneurysm        1110
non_aneurysm     190
Name: count, dtype: int64

LIA + RSNA
lesion
aneurysm    2400
Name: count, dtype: int64

LIA + RSNA with FPs:
lesion
aneurysm        2400
non_aneurysm     583
Name: count, dtype: int64

LIA + RSNA with FPs and GTs:
lesion
aneurysm        2574
non_aneurysm     583
Name: count, dtype: int64


In [1]:
lia_base = 1373
lia_with_fps = 393+1373
lia_with_fps_and_gts = 1464+393

rsna_base = 1027
rsna_base_with_fps = 1027+190
rsna_base_with_fps_and_gts = 1110+190

iters_with_fps = 22000

#  rule of for number of elements
iters_base = iters_with_fps * lia_base // lia_with_fps
iters_with_fps_and_gts = iters_with_fps * lia_with_fps_and_gts // lia_with_fps



iters_rsna_base = iters_with_fps * rsna_base // lia_with_fps
iters_rsna_with_fps_and_gts = iters_with_fps * rsna_base_with_fps_and_gts // lia_with_fps
iters_rsna_with_fps = iters_with_fps * rsna_base_with_fps // lia_with_fps
iters_base, iters_with_fps_and_gts, iters_rsna_base, iters_rsna_with_fps, iters_rsna_with_fps_and_gts

(17104, 23133, 12793, 15160, 16194)

In [2]:
iters_with_fps_and_gts

23133

In [3]:
lia_base = 1373
lia_with_fps = 393+1373
lia_with_fps_and_gts = 1464+393

rsna_base = 1027
rsna_base_with_fps = 1027+190
rsna_base_with_fps_and_gts = 1110+190

lia_rsna_base = 1373+1027
lia_rsna_with_fps = 1373+393+1027+190
lia_rsna_with_fps_and_gts = 1464+393+1110+190

iters_with_fps = 18000

#  rule of for number of elements
iters_base = iters_with_fps * lia_base // rsna_base
iters_with_fps_and_gts = iters_with_fps * lia_with_fps_and_gts // rsna_base
iters_base_with_fps = iters_with_fps * lia_with_fps // rsna_base


iters_rsna_base = iters_with_fps * rsna_base // rsna_base
iters_rsna_with_fps_and_gts = iters_with_fps * rsna_base_with_fps_and_gts // rsna_base
iters_rsna_with_fps = iters_with_fps * rsna_base_with_fps // rsna_base

iters_lia_rsna = iters_with_fps * lia_rsna_base // rsna_base
iters_lia_rsna_with_fps = iters_with_fps * lia_rsna_with_fps // rsna_base
iters_lia_rsna_with_fps_and_gts = iters_with_fps * lia_rsna_with_fps_and_gts // rsna_base

iters_base, iters_with_fps_and_gts, iters_rsna_base, iters_rsna_with_fps, iters_rsna_with_fps_and_gts

(24064, 32547, 18000, 21330, 22784)

In [4]:
iters_base_with_fps

30952

In [12]:
iters_lia_rsna, iters_lia_rsna_with_fps, iters_lia_rsna_with_fps_and_gts

(42064, 52282, 55332)

In [2]:
42/52, 2400/(584+2400)

(0.8076923076923077, 0.8042895442359249)

In [ ]:
# TODO:
# 1, 1, 0, 0, 0
# 1, 1, 1

## Remove only annotated aneurysms 

In [26]:
def remove_aneurysms_but_keep_non_reviewed(row):
    # if jisoo marked as aneurysm then remove
    if row["is_aneurysm"] == 1:
        return False
    # if jisoo marked as non-aneurysm then keep
    elif row["is_FP"] == "1" or row["is_FP"] == "1P":
        return True
    # if jisoo marked as infundibulum then discard
    elif row["is_infundibulum"] == 1:
        return True
    # if either of these are NaN, we need to check the Aneurysm column
    else:
        return True

last_valid_index = df_fps_og[
    ["is_aneurysm", "is_infundibulum", "is_FP"]
].dropna(how="all").index[-1]

df_fps_reviewed = df_fps_og.iloc[: last_valid_index + 1]
# remove all entries from reviewed that are None for all of the three columns
df_fps_reviewed = df_fps_reviewed.dropna(how="all", subset=["is_aneurysm", "is_infundibulum", "is_FP"]) 
df_fps_unreviewed = df_fps_og.iloc[last_valid_index + 1 :]

df_fps_reviewed["keep"] = df_fps_reviewed.apply(remove_aneurysms_but_keep_non_reviewed, axis=1)
df_fps_unreviewed["keep"] = True 
df_fps_reviewed = df_fps_reviewed[df_fps_reviewed["keep"] == True]
df_fps_unreviewed = df_fps_unreviewed[df_fps_unreviewed["keep"] == True]
df_mixed_fps = pd.concat([df_fps_reviewed, df_fps_unreviewed], ignore_index=True)
df_mixed_fps["lesion"] = "non_aneurysm"
# drop keep column 

df_mixed_fps = df_mixed_fps.drop(columns=["keep"])
len(df_mixed_fps)

/tmp/ipykernel_2751887/3283183182.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fps_unreviewed["keep"] = True


1304

In [27]:
total_discarded = len(df_fps_og) - len(df_mixed_fps)
print(f"Total discarded entries: {total_discarded}") 
total_kept = len(df_mixed_fps)
print(f"Total kept entries: {total_kept}")
total_fps_mixed = len(df_fps_unreviewed)
print(f"Total unreviewed entries kept: {total_fps_mixed}")

Total discarded entries: 106
Total kept entries: 1304
Total unreviewed entries kept: 908


In [31]:
cols_to_add = ["volume", "min_axis", "maj_axis", "iom_artery", "iom_vein"]
for col in cols_to_add:
    if col not in df_mixed_fps.columns:
        df_mixed_fps[col] = None
df_mixed_fps = df_mixed_fps[df.columns]

In [32]:
# combine with original df

df_train_mixed_fps = pd.concat([df, df_mixed_fps], ignore_index=True)
df_train_mixed_fps.to_csv("./fp_fix_nv/train_crop_0.4_mixed_fps.csv", index=False)

/tmp/ipykernel_2751887/2402565469.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_train_mixed_fps = pd.concat([df, df_mixed_fps], ignore_index=True)


In [38]:
path_rsna = "source_files_rsna/annotations_final.csv"
df_rsna = pd.read_csv(path_rsna)
df_rsna.columns = [
    "seriesuid",
    "coordX",
    "coordY",
    "coordZ",
    "d",
    "h",
    "w",
    "lesion",
    "volume",
    "maj_axis",
    "min_axis",
]
df_combined_non_r = df_combined_non_r[df_rsna.columns]

df_final = pd.concat([df_combined_non_r, df_rsna], ignore_index=True)
df_final.to_csv("./internal_train_crop_0.4_mixed_fps_done_rsna.csv", index=False)

In [39]:
df_rsna.head(1)

,seriesuid,coordX,coordY,coordZ,d,h,w,lesion,volume,maj_axis,min_axis
0,1.2.826.0.1.3680043.8.498.10005158603912009425...,283.0,275.0,162.0,6.740605,8.435049,9.386142,aneurysm,533.670969,9.386142,6.740605


In [40]:
df_combined_non_r.head(1).iloc[:, 0:20]

,seriesuid,coordX,coordY,coordZ,d,h,w,lesion,volume,maj_axis,min_axis
0,Tr0001.nii.gz,253.0,231.0,141.0,13.0,21.0,19.0,aneurysm,1499.0,21.317637,12.411585


In [41]:
def remove_aneurysms_and_not_reviewed(row):
    # if jisoo marked as aneurysm then remove
    if row["is_aneurysm"] == 1:
        return False
    # if jisoo marked as non-aneurysm then keep
    elif row["is_FP"] == "1" or row["is_FP"] == "1P":
        return True
    # if jisoo marked as infundibulum then discard
    elif row["is_infundibulum"] == 1:
        return True
    # if either of these are NaN, we need to check the Aneurysm column
    if (
        pd.isna(row["is_aneurysm"])
        and pd.isna(row["is_FP"])
        and pd.isna(row["is_infundibulum"])
    ):
        return False


df_fps = df_fps_og.copy()
df_fps["keep"] = df_fps.apply(remove_aneurysms_and_not_reviewed, axis=1)
df_fps_non_r = df_fps[df_fps["keep"]]
df_fps_non_r["lesion"] = "non_aneurysm"
df_fps_non_r

/tmp/ipykernel_951033/1121060062.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fps_non_r["lesion"] = "non_aneurysm"


,seriesuid,probability,coordX,coordY,coordZ,d,h,w,is_fp,Comments,Aneurysm,Jisoo review (y-if agreed w anatomy),comments,GY Comments,is_FP,is_aneurysm,is_infundibulum,needs_review,keep,lesion
0,Tr0001.nii.gz,0.992386,264.82675,304.70288,55.607536,15.581342,12.330435,12.551337,1.0,calcification of RV4,RICA-C7,y,NaN,NaN,1,NaN,NaN,NaN,True,non_aneurysm
1,Tr0001.nii.gz,0.945425,207.39828,205.01656,160.953900,5.864290,8.284704,8.960328,1.0,branch of RMCA-M1,NaN,y,NaN,NaN,1,NaN,NaN,NaN,True,non_aneurysm
2,Tr0002.nii.gz,0.972559,236.02684,288.35358,130.364720,12.565471,14.651035,14.723995,1.0,branch of right Posterior Inferior Cerebellar ...,RICA-C7,vein in post fossa,NaN,NaN,1,NaN,NaN,NaN,True,non_aneurysm
3,Tr0002.nii.gz,0.963586,347.29272,223.22868,8.606861,37.163760,30.523613,29.429272,1.0,not found (Left ICA-C2),NaN,y,NaN,NaN,1,NaN,NaN,NaN,True,non_aneurysm
4,Tr0002.nii.gz,0.948249,367.75235,225.88980,51.977787,9.012156,11.282431,11.766287,1.0,vessel of left face,NaN,y,NaN,NaN,1,NaN,NaN,NaN,True,non_aneurysm
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,Tr0358.nii.gz,0.996608,253.70757,303.55060,86.311676,25.603682,21.306190,21.753223,1.0,NaN,NaN,NaN,NaN,Extra cranial carotid internal,1P,NaN,NaN,NaN,True,non_aneurysm
497,Tr0358.nii.gz,0.915927,354.58620,229.98796,160.848820,5.785738,7.962148,8.748288,1.0,NaN,NaN,NaN,NaN,"False positive of the left crinoid ICA, somewh...",1P,NaN,NaN,NaN,True,non_aneurysm
498,Tr0360.nii.gz,0.870993,347.98236,200.60887,138.641770,5.588859,8.082442,9.058777,1.0,NaN,NaN,NaN,NaN,"False positive at the left MCA m1 m2 junction,...",1P,NaN,NaN,NaN,True,non_aneurysm
499,Tr0360.nii.gz,0.853835,308.04750,291.43103,1.991226,12.270049,20.687748,20.290630,1.0,NaN,NaN,NaN,NaN,Extra cranial,1P,NaN,NaN,NaN,True,non_aneurysm


In [42]:
cols_to_add = ["volume", "min_axis", "maj_axis", "iom_artery", "iom_vein"]
for col in cols_to_add:
    if col not in df_fps_non_r.columns:
        df_fps_non_r[col] = None
df_fps_non_r = df_fps_non_r[df.columns]
df_fps_non_r

/tmp/ipykernel_951033/4186681180.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fps_non_r[col] = None
/tmp/ipykernel_951033/4186681180.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fps_non_r[col] = None
/tmp/ipykernel_951033/4186681180.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.h

,seriesuid,coordX,coordY,coordZ,w,h,d,lesion,volume,min_axis,maj_axis,iom_artery,iom_vein
0,Tr0001.nii.gz,264.82675,304.70288,55.607536,12.551337,12.330435,15.581342,non_aneurysm,None,None,None,None,None
1,Tr0001.nii.gz,207.39828,205.01656,160.953900,8.960328,8.284704,5.864290,non_aneurysm,None,None,None,None,None
2,Tr0002.nii.gz,236.02684,288.35358,130.364720,14.723995,14.651035,12.565471,non_aneurysm,None,None,None,None,None
3,Tr0002.nii.gz,347.29272,223.22868,8.606861,29.429272,30.523613,37.163760,non_aneurysm,None,None,None,None,None
4,Tr0002.nii.gz,367.75235,225.88980,51.977787,11.766287,11.282431,9.012156,non_aneurysm,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,Tr0358.nii.gz,253.70757,303.55060,86.311676,21.753223,21.306190,25.603682,non_aneurysm,None,None,None,None,None
497,Tr0358.nii.gz,354.58620,229.98796,160.848820,8.748288,7.962148,5.785738,non_aneurysm,None,None,None,None,None
498,Tr0360.nii.gz,347.98236,200.60887,138.641770,9.058777,8.082442,5.588859,non_aneurysm,None,None,None,None,None
499,Tr0360.nii.gz,308.04750,291.43103,1.991226,20.290630,20.687748,12.270049,non_aneurysm,None,None,None,None,None


In [43]:
df_combined_non_r = pd.concat([df, df_fps_non_r], ignore_index=True)
df_combined_non_r.to_csv(
    "./internal_train_crop_0.4_confirmed_fps_done.csv", index=False
)

/tmp/ipykernel_951033/2098193903.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined_non_r = pd.concat([df, df_fps_non_r], ignore_index=True)


In [44]:
path_rsna = "source_files_rsna/annotations_final.csv"
df_rsna = pd.read_csv(path_rsna)
df_rsna.columns = [
    "seriesuid",
    "coordX",
    "coordY",
    "coordZ",
    "d",
    "h",
    "w",
    "lesion",
    "volume",
    "maj_axis",
    "min_axis",
]
df_combined_non_r = df_combined_non_r[df_rsna.columns]

df_final = pd.concat([df_combined_non_r, df_rsna], ignore_index=True)
df_final.to_csv("./internal_train_crop_0.4_confirmed_fps_done_rsna.csv", index=False)

In [45]:
path_og = "./internal_train_crop_0.4.csv"
path_fps = "./fps_fixed_done.csv"

df = pd.read_csv(path_og)

df_fps = df_fps_og.copy()

In [46]:
def keep_aneurysms_and_keep_non_reviewed(row):
    # if jisoo marked as aneurysm then remove
    if row["is_aneurysm"] == 1:
        return "aneurysm"
    # if jisoo marked as non-aneurysm then keep
    elif row["is_FP"] == "1" or row["is_FP"] == "1P":
        return "non_aneurysm"
    # if jisoo marked as infundibulum then discard
    elif row["is_infundibulum"] == 1:
        return "non_aneurysm"
    # if either of these are NaN, we need to check the Aneurysm column
    else:
        return "non_aneurysm"


df_fps = df_fps_og.copy()
df_fps["lesion"] = df_fps.apply(keep_aneurysms_and_keep_non_reviewed, axis=1)
df_fps_non_r = df_fps.copy()
cols_to_add = ["volume", "min_axis", "maj_axis", "iom_artery", "iom_vein"]
for col in cols_to_add:
    if col not in df_fps_non_r.columns:
        df_fps_non_r[col] = None

df_fps_non_r = df_fps_non_r[df.columns]
df_fps_non_r["lesion"].value_counts()

lesion
non_aneurysm    1319
aneurysm          91
Name: count, dtype: int64

In [47]:
df_fps["is_aneurysm"].value_counts()

is_aneurysm
1.0    91
Name: count, dtype: int64

In [48]:
df_combined_non_r = pd.concat([df, df_fps_non_r], ignore_index=True)
df_combined_non_r.to_csv(
    "./internal_train_crop_0.4_fps_done_extra_aneu.csv", index=False
)

/tmp/ipykernel_951033/1178979446.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined_non_r = pd.concat([df, df_fps_non_r], ignore_index=True)


In [50]:
path_rsna = "source_files_rsna/annotations_final.csv"
df_rsna = pd.read_csv(path_rsna)
df_rsna.columns = [
    "seriesuid",
    "coordX",
    "coordY",
    "coordZ",
    "d",
    "h",
    "w",
    "lesion",
    "volume",
    "maj_axis",
    "min_axis",
]
df_combined_non_r = df_combined_non_r[df_rsna.columns]

df_final = pd.concat([df_combined_non_r, df_rsna], ignore_index=True)
df_final.to_csv(
    "./internal_train_crop_0.4_mixed_fps_done_extra_aneu_rsna.csv", index=False
)

In [51]:
import pandas as pd
import numpy as np

In [52]:
path_og = "./internal_train_crop_0.4.csv"
path_fps = "./fps_fixed_done.csv"

df = pd.read_csv(path_og)
df_fps_og = pd.read_csv(path_fps).iloc[:, 2:20]
df_fps = df_fps_og.copy()

/tmp/ipykernel_951033/2281205112.py:5: DtypeWarning: Columns (11,12,13,14,15,16,19,20,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df_fps_og = pd.read_csv(path_fps).iloc[:, 2:20]


In [53]:
def keep_aneurysms_and_keep_only_reviewed(row):
    # if jisoo marked as aneurysm then remove
    if row["is_aneurysm"] == 1:
        return "aneurysm"
    # if jisoo marked as non-aneurysm then keep
    elif row["is_FP"] == "1" or row["is_FP"] == "1P":
        return "non_aneurysm"
    # if jisoo marked as infundibulum then discard
    elif row["is_infundibulum"] == 1:
        return "non_aneurysm"
    # if either of these are NaN, we need to check the Aneurysm column
    else:
        return "remove"


df_fps = df_fps_og.copy()
df_fps["lesion"] = df_fps.apply(keep_aneurysms_and_keep_only_reviewed, axis=1)
df_fps_non_r = df_fps.copy()
cols_to_add = ["volume", "min_axis", "maj_axis", "iom_artery", "iom_vein"]
for col in cols_to_add:
    if col not in df_fps_non_r.columns:
        df_fps_non_r[col] = None

df_fps_non_r = df_fps_non_r[df.columns]
df_fps_non_r["lesion"].value_counts()

lesion
remove          923
non_aneurysm    396
aneurysm         91
Name: count, dtype: int64

In [54]:
df_fps_non_r = df_fps_non_r[df_fps_non_r["lesion"] != "remove"]
df_fps["is_aneurysm"].value_counts()
df_combined_non_r = pd.concat([df, df_fps_non_r], ignore_index=True)
df_combined_non_r.to_csv(
    "./internal_train_crop_0.4_confirmed_fps_done_extra_aneu.csv", index=False
)
# -*- coding: utf-8 -*-

/tmp/ipykernel_951033/3527990590.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined_non_r = pd.concat([df, df_fps_non_r], ignore_index=True)


In [55]:
path_rsna = "source_files_rsna/annotations_final.csv"
df_rsna = pd.read_csv(path_rsna)
df_rsna.columns = [
    "seriesuid",
    "coordX",
    "coordY",
    "coordZ",
    "d",
    "h",
    "w",
    "lesion",
    "volume",
    "maj_axis",
    "min_axis",
]
df_combined_non_r = df_combined_non_r[df_rsna.columns]

df_final = pd.concat([df_combined_non_r, df_rsna], ignore_index=True)
df_final.to_csv(
    "./internal_train_crop_0.4_confirmed_fps_done_extra_aneu_rsna.csv", index=False
)